In [ ]:
#Data Preprocessing

X = df.drop(columns="Delivery_Time_min")
y = df["Delivery_Time_min"]

categorical_features = [
    "Order_Priority", "Vehicle_Type", "Day_of_Week"
]

numeric_features = [
    "Distance_km", "Package_Weight_kg", "Traffic_Index",
    "Weather_Score", "Warehouse_Load_pct"
]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Exploratory Analysis Plan

print(df.head())
print(df.info())
print(df.describe())
print(df.isnull().sum())

print(df.groupby("Vehicle_Type")["Delivery_Time_min"].mean())
print(df.groupby("Order_Priority")["Delivery_Time_min"].mean())

# Python Implementation

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=8,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
                n_estimators=250,
        max_depth=12,
        min_samples_leaf=3,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

#Model Evaluation

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

#Cross-Validation

cv = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = np.sqrt(
    -cross_val_score(
        selected_pipeline,
        X, y,
        scoring="neg_mean_squared_error",
        cv=cv
    )
)

print("CV RMSE:", rmse_scores)
print("Mean CV RMSE:", rmse_scores.mean())
print("Std CV RMSE:", rmse_scores.std())

# Hyperparameter Tuning

from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [6, 8, 10, 12, None],
    "model__min_samples_leaf": [1, 2, 3, 5]
}

search = RandomizedSearchCV(
    selected_pipeline,
    param_distributions=param_grid,
    n_iter=15,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print(search.best_params_)
print(search.best_score_)


